# Perform balanced sampling

In [1]:
import pandas as pd

In [2]:
label_data = pd.read_parquet("../data/processed/chain/labeled_days.parquet")
summary_df = pd.read_csv("../data/processed/chain/label_days_summery.csv")

**check address which has more anomaly days than normal days**

In [3]:
summary_df['normal_days'] = summary_df['total_active_days'] - summary_df['anomaly_days']

addresses_anom_gt_norm = summary_df[
    summary_df['anomaly_days'] > summary_df['normal_days']
]

print(f"Number of addresses where anomalous > normal: {len(addresses_anom_gt_norm)}")
addresses_anom_gt_norm[['address', 'normal_days', 'anomaly_days']]


Number of addresses where anomalous > normal: 6


,address,normal_days,anomaly_days
95,0x078c2b3b09528fd7b80c9ee715f378d382f9139b,3,4
518,0x2b065809f6ec6df32878bcd26711a0e2bcf59c26,3,4
626,0x33d262cbb8d4bcc568f7809230ecf609dd31d379,5,6
641,0x35280a184f18a00b785875bcc848f1e227bed1ca,4,5
1363,0x699cbeaa28a49e096df00011d734803a0b8f5f80,3,4
2124,0xa29d8193370632e65f00e25a518230147e410660,3,4


**balancing the dataset**

In [4]:
balanced_samples = []

for address, group in label_data.groupby("address"):
    anomalous = group[group['is_anomalous'] == 1]
    normal = group[group['is_anomalous'] == 0]

    if len(anomalous) > len(normal):
        # Ignore anomalies, keep only normal rows
        balanced_addr_df = normal
    else:
        # Keep all anomalies and sample equal number of normals
        normal_sampled = normal.sample(n=len(anomalous), random_state=42)
        balanced_addr_df = pd.concat([anomalous, normal_sampled])

    balanced_samples.append(balanced_addr_df)

# Combine all addresses
balanced_per_address_df = (
    pd.concat(balanced_samples)
      .sample(frac=1, random_state=42)  # shuffle
      .reset_index(drop=True)
)

print(f"Total days: {len(balanced_per_address_df)}")
print(balanced_per_address_df['is_anomalous'].value_counts())
print(balanced_per_address_df['is_anomalous'].value_counts(normalize=True) * 100)

# Sort by day
balanced_per_address_df = balanced_per_address_df.sort_values('day').reset_index(drop=True)

output_parquet = "../data/processed/chain/balanced_label_days.parquet"
balanced_per_address_df.to_parquet(output_parquet, index=False)
print(f"Saved balanced dataset to: {output_parquet}")

csv_path = "../data/processed/chain/balanced_label_days.csv"
balanced_per_address_df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")


Total days: 66557
is_anomalous
0    33289
1    33268
Name: count, dtype: int64
is_anomalous
0    50.015776
1    49.984224
Name: proportion, dtype: float64
Saved balanced dataset to: ../data/processed/chain/balanced_label_days.parquet
Saved CSV to: ../data/processed/chain/balanced_label_days.csv
